# 07 — Reading a JINA/libnucnet database

`read_jina_xml` takes a nuclide file, a reaction file, and optionally a zone
file. This notebook builds miniature versions of each so it runs without any
external download; notebook 08 uses a full production database.

In [ ]:
# The notebooks run against the installed package.  If you are working from a
# checkout without installing, uncomment the two lines below.
# import sys, pathlib
# sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nucnetpy as nn

print("nucnetpy version:", nn.__version__)

## Miniature JINA-style files

In [ ]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)

Path("data/toy_jina_nuclides.xml").write_text('''<?xml version="1.0"?>
<nuclear_data>
  <nuclide>
    <z>2</z><a>4</a><source>ame</source>
    <mass_excess>2.424915</mass_excess><spin>0</spin>
  </nuclide>
  <nuclide>
    <z>6</z><a>12</a><source>ame</source>
    <mass_excess>0.0</mass_excess><spin>0</spin>
    <partf_table>
      <point><t9>1.0</t9><log10_partf>0.0</log10_partf></point>
      <point><t9>10.0</t9><log10_partf>0.2</log10_partf></point>
    </partf_table>
  </nuclide>
  <nuclide>
    <z>8</z><a>16</a><source>ame</source>
    <mass_excess>-4.737</mass_excess><spin>0</spin>
  </nuclide>
</nuclear_data>
''')

Path("data/toy_jina_reactions.xml").write_text('''<?xml version="1.0"?>
<reaction_data>
  <reaction>
    <source>demo</source>
    <reactant>c12</reactant><reactant>he4</reactant>
    <product>o16</product><product>gamma</product>
    <non_smoker_fit>
      <a1>10.0</a1><a2>0.0</a2><a3>0.0</a3><a4>0.0</a4>
      <a5>0.0</a5><a6>0.0</a6><a7>0.0</a7><a8>0.0</a8>
    </non_smoker_fit>
  </reaction>
  <reaction>
    <source>demo</source>
    <reactant>o16</reactant><reactant>gamma</reactant>
    <product>c12</product><product>he4</product>
    <non_smoker_fit>
      <a1>5.0</a1><a2>0.0</a2><a3>0.0</a3><a4>0.0</a4>
      <a5>0.0</a5><a6>0.0</a6><a7>0.0</a7><a8>0.0</a8>
    </non_smoker_fit>
  </reaction>
</reaction_data>
''')
print("written")

## Read and validate

`validate()` reports three separate things. `missing_species` are named by a
reaction but absent from the species map; `invalid_reactions` do not balance in
mass or charge; `species_without_nuclear_data` were synthesised because the
reaction file mentioned them and the nuclide file did not. The last is the one
that quietly ruins equilibrium calculations.

In [ ]:
net = nn.read_jina_xml("data/toy_jina_nuclides.xml",
                      "data/toy_jina_reactions.xml")
report = net.validate()

# net.species holds nuclides only.  species_names() additionally reports every
# participant named by a reaction, so the photon appears there and not in the
# species map -- it is a participant, not a nuclide.
print("nuclides:    ", len(net.species), sorted(net.species))
print("participants:", net.species_names())
print("reactions:   ", len(net.reactions.reactions))
for key in ["missing_species", "invalid_reactions", "species_without_nuclear_data"]:
    print(f"  {key}: {report[key]}")

In [ ]:
for r in net.reactions.reactions:
    print(f"{r.string:28s} order={r.reactant_order}  "
          f"rate(T9=2) = {r.rate(2.0):.4e}")

The photodisintegration is a one-body process: `gamma` appears in
the record but not in the reactant order, exactly as described in notebook 03.

## Cutting a database down to size

A production database has thousands of nuclides, far more than a one-zone
calculation needs. `select_species` picks by charge and mass number, and
`limit_network` keeps only reactions all of whose participants survive.

**Keep `gamma` in the list.** It is not a nuclide, so `select_species` will not
return it, and without it every photodisintegration is discarded.

In [ ]:
from nucnetpy.network_limiter import select_species, limit_network
import copy

for keep_photon in [False, True]:
    trial = copy.deepcopy(net)
    wanted = select_species(trial, zmax=8)
    limit_network(trial, wanted + (["gamma"] if keep_photon else []))
    print(f"keep gamma = {str(keep_photon):5s} -> "
          f"{len(trial.reactions.reactions)} reaction(s) retained")

## Combining into one file

In [ ]:
nn.combine_jina_xml("data/toy_jina_nuclides.xml",
                    "data/toy_jina_reactions.xml",
                    "data/combined_toy_network.xml")
print(Path("data/combined_toy_network.xml").read_text()[:400])